In [1]:
import os
import json
import datetime
import warnings
import polars as pl
import pandas as pd
import altair as alt

from src.najdi_rok import najdi_rok
from src.pocet_stran import pocet_stran
from src.bez_bordelu import bez_bordelu
from src.alt_friendly import alt_friendly
from src.hezke_jmeno import hezke_jmeno
from src.kristi_promin import kristi_promin
from src.me_to_neurazi import me_to_neurazi
from src.alt_friendly import alt_friendly

with open(os.path.join('src','kredity.json'), 'r', encoding='utf-8') as kredity:
    kredity = json.loads(kredity.read())
pl.Config(tbl_rows=100)
alt.data_transformers.disable_max_rows()
alt.themes.register('irozhlas', kristi_promin)
alt.themes.enable('irozhlas')
warnings.filterwarnings('ignore')

In [2]:
df = pl.read_parquet(os.path.join("data/cnb_sloupce","100.parquet"))
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","leader.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","008.parquet")), left_on="001", right_on="001", how="left")
df = df.to_pandas()
df = df[df["leader"].str[6].isin(["a", "t"])]
df = df[~df["leader"].str[7].isin(["b", "i", "s", " "])]
df = df[(df["008"].str[15:17] == "xr") & (df["008"].str[35:38] == "cze")]
df = pl.from_pandas(df)
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","022.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","245.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","300.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","655.parquet")), left_on="001", right_on="001", how="left")
df = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","700.parquet")), left_on="001", right_on="001", how="left")
df = df.explode("022_a").filter(pl.col("022_a").is_null())
df = df.with_columns(pl.col('008').map_elements(najdi_rok, return_dtype=int).alias('rok'))
df = df.with_columns(pl.col('300_a').map_elements(pocet_stran, return_dtype=int).alias('stran'))
df = df.with_columns(pl.col('245_a').map_elements(bez_bordelu, return_dtype=str))
df = df.explode('245_p').with_columns(pl.col('245_p').map_elements(bez_bordelu, return_dtype=str))
print(len(df))

716789


In [3]:
aut = pl.read_parquet(os.path.join("data","aut_vyber.parquet"))

In [4]:
df = df.filter(pl.col("rok") >= 1800)

In [5]:
print(len(df))

710142


In [6]:
df = df.filter((~pl.col("245_h").str.contains("grafika")) | pl.col("245_h").is_null()).unique(subset=["008","100_a","245_a","245_p"], keep="first")

In [7]:
print(len(df))

705921


In [8]:
nechcemejetam = [
    "jn20001103401",
    "xx0008006",
    "jn19990008769",
    "jx20060515016",
    "jn19981002230",
    "jn19981001737",
    "jn19990210182",
    "jn19990001842",
    "jn19990002786",
    "jn19990004346",
    "jn20020721077",
    "jn19990210513",
    "jn19990005488",
    "jo20000080627",
    "jn19990000171",
    "jn20001005715",
    "jn19981002409",
    "jn20000810141",
    "jn19981002129",
    "jn20001103529",
    "jn20000810032",
    "jn19990001513",
    "jx20040611003",
    "jn19990005499",
    "jn19981002230",
    "jn19990001907",
    "jo2005267810",
    "jo20241218643",
    "jn20000602144",
    "jn19990005454",
    "jn19981228078",
    "xx0010566",
    "jn20010310318",
    "xx0203193",
    "jn20001227589",
    "jo2003204270",
    "xx0082647",
    "jn19990001239"
]

# df = df.filter(~pl.col("100_7").is_in(nechcemejetam))

In [9]:
df.sample(10)

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64
"""1""","""O'Neill, Poppy""","""mzk20211121725""","[""aut""]",null,null,null,null,null,"""nkc20213330536""",""" nam a22 i 4500""","""210707s2021 xr a b f 0…",null,null,null,null,null,"""1""","""0""","""Buď sám sebou""","""nech svoje já zazářit /""","""Poppy O'Neill ; překlad: Petra…",null,null,null,null,null,"[""139 stran :""]","[""ilustrace ;""]","[""21 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""příručky"", ""pracovní sešity"", … ""children's literature""]","[""fd133209"", ""fd133116"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","[""Šmídová, Petra""]","[""trl""]",null,"[""xx0149141""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2021,139
"""1""","""Miller, Jan,""","""jk01081712""","[""aut""]","""1870-1937""",null,null,null,null,"""cpk20233536852""",""" cam a22 aa4500""","""001015m19261929xr |…",null,null,null,null,null,"""1""","""0""","""Vesnické humoresky""",null,"""Jan Miller""",null,null,null,null,null,"[""2 sv. (195; 132 s.) ;""]",null,"[""18 cm""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1926,195
"""1""","""Navrátilová, Jana""","""jn20001103670""","[""aut""]",null,null,null,null,null,"""nkc20051578784""",""" nam a22 a 4500""","""050817s2005 xr g f 0…",null,null,null,null,null,"""1""","""0""","""Ruština""","""konverzace & slovník = češsko-…","""Jana Navrátilová""",null,null,null,null,null,"[""288 s. ;""]",null,"[""12 cm""]",null,null,null,"[""7"", ""9""]","[""příručky"", ""handbooks and manuals""]","[""fd133209"", null]","[""czenas"", ""eczenas""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2005,288
"""1""","""Wessel, Kathrin,""","""xx0306459""","[""ill""]","""1981-""",null,null,null,null,"""nkc20233546709""",""" nam a22 i 4500""","""230907t20232023xr a a 0…",null,null,null,null,null,"""1""","""0""","""Když se statek probouzí""",null,"""Kathrin Wessel, Sandra Grimm""",null,null,null,null,null,"[""16 nečíslovaných stran :""]","[""barevné ilustrace ;""]","[""18 x 23 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""leporela"", ""publikace pro děti"", … ""children's literature""]","[""fd132727"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","[""Grimm, Sandra,""]","[""aut""]","[""1974-""]","[""xx0148949""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2023,16
"""1""","""Pierazzi Mitri, Monica""","""ola2012729333""","[""ill""]",null,null,null,null,null,"""nkc20122412477""",""" nam a22 a 4500""","""121023s2012 xr a a 0…",null,null,null,null,null,"""1""","""0""","""Učíme se počítat""","""se samolepkami /""","""[ilustrace Monica Pierazzi Mit…",null,null,null,null,null,"[""[36] s. :""]","[""barev. il. ;""]","[""21 cm +""]","[""samolepky ([2] l.)""]",null,

In [10]:
df = df.drop_nulls(subset=["700_a","700_4","700_7"]).explode(["700_a","700_4","700_7"])

In [11]:
df.select(pl.col(["700_a","700_4","700_7",'245_a'])).sample(10)

700_a,700_4,700_7,245_a
str,str,str,str
"""Jerie, Alexander,""","""trl""","""ola2003209130""","""Pirueta"""
"""Němcová, Edda""","""trl""","""mzk2005266569""","""Královský rubín"""
"""Vyhnálek, Oldřich""","""trl""",null,"""Tekuté slunce"""
"""Havlíček, Miloslav,""","""bjd""","""jo2003204261""","""Khanův hněv"""
"""Marlier, Marcel,""","""ill""","""jn20000810202""","""Martinka se učí kreslit"""
"""Škodný, David,""","""ill""","""ola20221152275""","""Příběhy ze Šumavy v komiksech"""
"""Steinerová, Irena,""","""trl""","""jn20010309566""","""Na útěku"""
"""Dvořák, Libor,""","""trl""","""jk01030170""","""Vojna a mír"""
"""Ševicová, Martina""","""trl""",null,"""Vláčky"""


## Autorské spolupráce

In [13]:
from itertools import combinations

In [14]:
def find_collaborations(df):
    # Filter for authors only
    authors = df.filter(pl.col('700_4') == 'aut')
    
    # Group by book title to get authors per book
    books_authors = authors.group_by('245_a').agg(pl.col('700_a').alias('authors'))
    
    # Generate author pairs and count collaborations
    collaborations = []
    for book in books_authors.iter_rows():
        if len(book[1]) > 1:  # Only consider books with multiple authors
            for pair in combinations(sorted(book[1]), 2):
                collaborations.append(pair)
    
    # Convert to dataframe and count frequencies
    collab_df = pl.DataFrame({
        'author1': [p[0] for p in collaborations],
        'author2': [p[1] for p in collaborations]
    })
    
    if len(collab_df) == 0:
        return pl.DataFrame({'author1': [], 'author2': [], 'collaboration_count': []})
    
    return (collab_df.group_by(['author1', 'author2'])
            .count()
            .sort('count', descending=True)
            .rename({'count': 'collaboration_count'}))

In [15]:
find_collaborations(df).filter(pl.col("author1") != pl.col("author2"))

author1,author2,collaboration_count
str,str,u32
"""Novotný, Miloš""","""Novák, František""",1165
"""Král, Lukáš""","""Valenta, Tomáš""",1156
"""Krupka, Peter,""","""Nechvátalová, Jana,""",1089
"""Malý, Martin""","""Münch, Otto""",897
"""Münch, Otto""","""Čechová, Jarmila""",826
"""Frydryšková, Yvetta""","""Münch, Otto""",820
"""Krupka, Peter,""","""Staudková, Hana""",792
"""Nechvátalová, Jana,""","""Staudková, Hana""",792
"""Medek, Jaroslav,""","""Petrůj, Svatopluk,""",761


In [16]:
df_autorske = df.filter(pl.col('700_4') == 'aut')

In [17]:
def hezkejmeno(sto):
    if not sto[-1].isalnum():
        sto = sto[:-1]
    if "," in sto:
        sto = sto.split(",")
        sto = sto[1].strip() + " " + sto[0].strip()
    return sto    

In [18]:
df_autorske = df_autorske.with_columns(pl.col("100_a").map_elements(hezkejmeno).alias("jmeno1")).with_columns(pl.col("700_a").map_elements(hezkejmeno).alias("jmeno2"))

In [19]:
df_autorske

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str
"""1""","""Nesvadba, Antonín""","""jk01083229""","[""aut""]",null,null,null,null,null,"""ck8501596""",""" nam a22 4500""","""850423s1985 xr a u0…",null,null,null,null,null,"""1""","""0""","""Úloha elektroniky při intenziv…",null,"""Antonín Nesvadba, Pavel Schrán…",null,null,null,null,null,"[""115 s. :""]","[""tb., schémata ;""]","[""20 cm""]",null,null,null,null,null,null,null,null,null,null,"[""1""]","""Schránil, Pavel,""","""aut""","[""1933-""]","""mzk2002112067""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1985,115,"""Antonín Nesvadba""","""Pavel Schránil"""
"""1""","""Jentschura, Peter,""","""mzk2009517324""","[""aut""]","""1941-""",null,null,null,null,"""nkc20091929452""",""" nam a22 a 4500""","""090512s2009 xr a e f 0…",null,null,null,null,null,"""1""","""0""","""Zdraví díky odkyselení a vylou…","""rozpouštění usazenin, neutrali…","""Peter Jentschura, Josef Lohkäm…",null,null,null,null,null,"[""238 s. :""]","[""il. ;""]","[""24 cm""]",null,null,null,"[""7"", ""9""]","[""populárně-naučné publikace"", ""popular works""]","[""fd131864"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1""]","""Lohkämper, Josef,""","""aut""","[""1930-""]","""mzk2009517325""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2009,238,"""Peter Jentschura""","""Josef Lohkämper"""
"""1""","""Abrahámová, Jitka,""","""jn20001103578""","[""aut""]","""1943-""",null,null,null,null,"""cpk20000983051""",""" nam a22 a 4500""","""010130s2000 xr a e 0…",null,null,null,null,null,"""1""","""0""","""Rakovina tlustého střeva a kon…",null,"""Jitka Abrahámová, Ludmila Boub…",null,null,null,null,null,"[""20 s. :""]","[""il. ;""]","[""16 cm""]",null,null,null,"[""7""]","[""informační publikace""]","[""fd132454""]","[""czenas""]",null,null,null,"[""1"", ""1""]","""Boublíková, Ludmila""","""aut""",null,"""jn20010310080""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2000,20,"""Jitka Abrahámová""","""Ludmila Boublíková"""
"""1""","""Abrahámová, Jitka,""","""jn20001103578""","[""aut""]","""1943-""",null,null,null,null,"""cpk20000983051""",""" nam a22 a 4500""","""010130s2000 xr a e 0…",null,null,null,null,null,"""1""","""0""","""Rakovina tlustého střeva a kon…",null,"""Jitka Abrahámová, Ludmila Boub…",null,null,null,null,null,"[""20 s. :""]","[""il. ;""]","[""16 cm""]",null,null,null,"[""7""]","[""informační publikace""]","[""fd132454""]","[""czenas""]",null,null,null,"[""1"", ""1""]","""Kordíková, Drahomíra""","""aut""",null,"""jn20010310081""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2000,20,"""Jitka Abrahámová""","""Drahomíra Kordíková"""
"""0""","""Peyo,""","""xx0007049""","[""ill""]","""1928-1992""",null,null,null,null,"""cpk20031244869""",""" nam a22 a 4500""","""030610s2003 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Šmoulí vynálezy""",null,"""Peyo ; napsali Philippe Del

In [20]:
def kombajmen(jmeno1, jmeno2, radit=True, spojeni = "a"):
    try:
        jmena = [jmeno1.split(" ")[-1], jmeno2.split(" ")[-1]]
        if radit == True:
            jmena.sort()
        return f" {spojeni} ".join(jmena)
    except Exception as e:
        print(e)
        return None

In [21]:
df_autorske = df_autorske.with_columns(pl.struct('jmeno1','jmeno2').map_elements(lambda x: kombajmen(x['jmeno1'], x['jmeno2'])).alias("dvojice"))

In [22]:
df_autorske.group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

dvojice,245_a
str,u32
"""David a Soukup""",126
"""Nedbalová a Nedbalová""",90
"""Engels a Marx""",60
"""Benešová a Kanyzová""",52
"""Benešová a Fukalová""",52
"""Benešová a Formáčková""",52
"""Benešová a Šmíd""",52
"""Benešová a Příhoda""",52
"""Benešová a Vondrák""",52


In [23]:
df_autorske.group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

100_a,100_7,700_a,245_a
str,str,str,u32
"""David, Petr,""","""xx0006786""","""Soukup, Vladimír,""",126
"""Nedbalová, Marie""","""jo2014816080""","""Nedbalová, Josefína""",87
"""Marx, Karl,""","""jn19990005454""","""Engels, Friedrich,""",59
"""Benešová, Hana,""","""pna2012702164""","""Fukalová, Lenka""",52
"""Benešová, Hana,""","""pna2012702164""","""Formáčková, Marie,""",52
"""Benešová, Hana,""","""pna2012702164""","""Šmíd, Václav,""",52
"""Benešová, Hana,""","""pna2012702164""","""Kanyzová, Žofie,""",52
"""Benešová, Hana,""","""pna2012702164""","""Vondrák, Jan,""",52
"""Benešová, Hana,""","""pna2012702164""","""Příhoda, Pavel,""",52


In [24]:
df.filter(pl.col("700_a") == "Soukup, Vladimír,").filter(pl.col("100_a") == "David, Petr,")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""nkc20122348626""",""" cam a22 a 4500""","""120807s2012 xr ab g f 0…",null,null,null,null,null,"""1""","""0""","""Hrady, zámky a tvrze""",null,"""Petr David, Vladimír Soukup""",null,null,null,null,null,"[""207 s. :""]","[""barev. il., mapy ;""]","[""25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""průvodce"", ""fotografické publikace"", … ""photographical works""]","[""fd133154"", ""fd132276"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Soukup, Vladimír,""","""aut""","[""1949-""]","""xx0006793""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2012,207
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""nkc20193115955""",""" cam a22 i 4500""","""190530t20192019xr ab e f 0…",null,null,null,null,null,"""1""","""0""","""Šumava""",null,"""Petr David, Petr Ludvík, Vladi…",null,null,null,null,null,"[""182 stran :""]","[""barevné ilustrace, mapy ;""]","[""22 cm""]",null,null,null,"[""7"", ""9""]","[""turistické průvodce"", ""tourist guidebooks""]","[""fd133738"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1"", ""1""]","""Soukup, Vladimír,""","""aut""","[""1977-"", ""1949-""]","""xx0006793""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2019,182
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""nkc20071708508""",""" nam a22 a 4500""","""070212s2006 xr ab g f 0…",null,null,null,null,null,"""1""","""0""","""Českokrumlovsko""",null,"""[autoři Petr David, Věra Dobro…",null,null,null,null,null,"[""135 s. :""]","[""il., mapy ;""]","[""20 cm""]",null,null,null,"[""7"", ""9""]","[""turistické průvodce"", ""tourist guidebooks""]","[""fd133738"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1"", ""1""]","""Soukup, Vladimír,""","""aut""","[""1947-"", ""1949-""]","""xx0006793""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2006,135
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""nkc20051627547""",""" cam a22 a 4500""","""050909s2005 xr ab g f 0…",null,null,null,null,null,"""1""","""0""","""Plzeňsko - jih""","""Plzeň /""","""[Petr David, Věra Dobrovolná, …",null,null,null,null,null,"[""191 s. :""]","[""il., mapy ;""]","[""20 cm""]",null,null,null,"[""7"", ""9""]","[""turistické průvodce"", ""tourist guidebooks""]","[""fd133738"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1"", ""1""]","""Soukup, Vladimír,""","""aut""","[""1947-"", ""1949-""]","""xx0006793""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2005,191
"""1""","""David, Petr,""","""xx0006786""","[""aut""]","""1949-""",null,null,null,null,"""nkc20132484598""",""" nam a22 ia4500""","""130822m20102016xr ab e f 0…",null,null,null,null,null,"""1""","""0""","""Špalíček v

## Ilustrátorstvo

In [26]:
df_ill = df.filter(pl.col('700_4') == 'ill')

In [27]:
df_ill = df_ill.with_columns(pl.col("100_a").map_elements(hezkejmeno).alias("jmeno1")).with_columns(pl.col("700_a").map_elements(hezkejmeno).alias("jmeno2"))

In [28]:
df_ill = df_ill.with_columns(pl.struct('jmeno1','jmeno2').map_elements(lambda x: kombajmen(x['jmeno1'], x['jmeno2'], radit=False, spojeni="&")).alias("dvojice"))

In [29]:
df_ill.filter(~pl.col("100_7").is_in(nechcemejetam)).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

dvojice,245_a
str,u32
"""Štíplová & Němeček""",108
"""Nedbalová & Popprová""",87
"""Nedbalová & Poppr""",68
"""Bass & Kratochvíl""",67
"""Švandrlík & Winter-Neprakta""",64
"""Čapek & Čapek""",33
"""Pospíšilová & Trsťan""",31
"""Hašek & Lada""",30
"""Rosecká & Růžička""",28


In [30]:
koliktohobylo = df_ill.unique(subset=['100_7','245_a']).filter(~pl.col("100_7").is_in(nechcemejetam)).filter(pl.col("stran") > 30).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

In [31]:
dvojice_aut_ill = df_ill.unique(subset=['100_7','245_a']).filter(~pl.col("100_7").is_in(nechcemejetam)).filter(pl.col("stran") > 30).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True).head(5).select(pl.col("dvojice")).to_series().to_list()
dvojice_aut_ill

['Štíplová & Němeček',
 'Švandrlík & Winter-Neprakta',
 'Pospíšilová & Trsťan',
 'Hašek & Lada',
 'Rosecká & Růžička']

In [32]:
df_ill.filter(~pl.col("100_7").is_in(nechcemejetam)).group_by(['dvojice']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

dvojice,245_a
str,u32
"""Štíplová & Němeček""",108
"""Nedbalová & Popprová""",87
"""Nedbalová & Poppr""",68
"""Bass & Kratochvíl""",67
"""Švandrlík & Winter-Neprakta""",64
"""Čapek & Čapek""",33
"""Pospíšilová & Trsťan""",31
"""Hašek & Lada""",30
"""Rosecká & Růžička""",28


In [33]:
import datetime

In [34]:
df_ill.filter(pl.col("dvojice") == "Pospíšilová & Trsťan")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2,dvojice
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str,str
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20142633308""",""" nam a22 a 4500""","""141016s2014 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Zlobivé pohádky""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""104 s. :""]","[""barev. il. ;""]","[""25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2014,104,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20162865008""",""" nam a22 i 4500""","""161222s2017 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Nezralá, Hruška, k tabuli!""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""72 stran :""]","[""barevné ilustrace ;""]","[""23 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2017,72,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20193112591""",""" nam a22 i 4500""","""190625s2019 xr a c 0…",null,null,null,null,null,"""1""","""0""","""Napiš správně dě, tě, ně, bě, …","""křížovky, osmisměrky, doplňova…","""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""112 stran :""]","[""ilustrace ;""]","[""29 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""pracovní sešity"", ""publikace pro děti"", … ""children's literature""]","[""fd133116"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2019,112,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20102158419""",""" nam a22 a 4500""","""110325s2011 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Máš nadání na hádání""",null,"""Zuzana Pospíšilová ; ilustrace…",null,null,null,null,null,"[""118 s. :""]","[""barev. il. ;""]","[""23 cm""]",null,null,null,"[""7"", ""9""]","[""publikace pro děti"", ""children's literature""]","[""fd133156"", null]","[""czenas"", ""eczen

In [35]:
do_grafu = df_ill.filter(
    pl.col("dvojice").is_in(dvojice_aut_ill)
).unique(
    subset=['245_a']
).join(
    koliktohobylo, how="left", on="dvojice"
).with_columns(
    pl.col("245_a_right").map_elements(lambda x: str(x) + "×")
).with_columns(
    pl.concat_str([pl.col('245_a_right'), pl.col('dvojice')], separator=' ').alias('dvojice')
)

In [36]:
dvojice_sort = do_grafu.group_by("dvojice").len().sort(by="len",descending=True).select(pl.col("dvojice")).to_series().to_list()
dvojice_sort

['105× Štíplová & Němeček',
 '62× Švandrlík & Winter-Neprakta',
 '30× Pospíšilová & Trsťan',
 '29× Hašek & Lada',
 '27× Rosecká & Růžička']

In [37]:
df_ill.filter(pl.col("dvojice") == "Štíplová & Němeček")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2,dvojice
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str,str
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""ck8705701""",""" nam a22 4500""","""871218s1986 xr a a 6 0…",null,null,null,null,null,"""1""","""0""","""Zámek bez klíče""",null,"""napsala Ljuba Štíplová ; nakre…",null,null,null,null,null,"[""34 s. :""]","[""barev. il. ;""]","[""23 cm""]",null,null,null,null,null,null,null,null,null,null,"[""1""]","""Němeček, Jaroslav,""","""ill""","[""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1986,34,"""Ljuba Štíplová""","""Jaroslav Němeček""","""Štíplová & Němeček"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""cpk20203176060""",""" cam a22 a 4500""","""051123s1975 xr j |…",null,null,null,null,null,"""1""","""0""","""Modrý přízrak a čtyři další ob…",null,"""napsala Ljuba Štíplová ; kresb…",null,null,null,null,null,"[""34 s. :""]","[""il.""]",null,null,null,null,null,null,null,null,null,null,null,"[""1""]","""Němeček, Jaroslav,""","""ill""","[""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1975,34,"""Ljuba Štíplová""","""Jaroslav Němeček""","""Štíplová & Němeček"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""cpk20233525018""",""" cam a22 a 4500""","""120619s1982 xr a j 0…",null,null,null,null,null,"""1""","""0""","""Pozor, padá omítka""",null,"""[napsala Ljuba Štíplová ... et…",null,null,null,null,null,"[""34 s. :""]","[""il.""]",null,null,null,null,"[""7"", ""7"", ""7""]","[""české prózy"", ""komiksy"", ""publikace pro děti""]","[""fd133972"", ""fd131978"", ""fd133156""]","[""czenas"", ""czenas"", ""czenas""]",null,null,null,"[""1""]","""Němeček, Jaroslav,""","""ill""","[""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1982,34,"""Ljuba Štíplová""","""Jaroslav Němeček""","""Štíplová & Němeček"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""bk197505365""",""" nam a22 1 4500""","""971124s1975 xr a a 6 0…",null,null,null,null,null,"""1""","""0""","""Vládci peřejí""","""a čtyři další obrázkové příběh…","""Ljuba Štíplová, Jaroslav Němeč…",null,null,null,null,null,"[""34, [1] s. :""]","[""barev. il. ;""]","[""8°""]",null,null,null,null,null,null,null,null,null,null,"[""1""]","""Němeček, Jaroslav,""","""ill""","[""1944-""]","""jk01083128""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1975,34,"""Ljuba Štíplová""","""Jaroslav Němeček""","""Štíplová & Němeček"""
"""1""","""Štíplová, Ljuba,""","""jk01131441""","[""aut""]","""1930-2009""",null,null,null,null,"""ck8705674""",""" nam a22 4500""","""871221s1987 xr a a 6 0…",null,null,null,null,null,"""1""","""0""","""Bílá past""",null,"""napsala Ljuba Štíplová ; nakre…",null,null,null

In [38]:
df_ill.filter(pl.col("dvojice") == "Pospíšilová & Trsťan")

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,jmeno1,jmeno2,dvojice
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64,str,str,str
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20142633308""",""" nam a22 a 4500""","""141016s2014 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Zlobivé pohádky""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""104 s. :""]","[""barev. il. ;""]","[""25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2014,104,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20162865008""",""" nam a22 i 4500""","""161222s2017 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Nezralá, Hruška, k tabuli!""",null,"""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""72 stran :""]","[""barevné ilustrace ;""]","[""23 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd133973"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2017,72,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20193112591""",""" nam a22 i 4500""","""190625s2019 xr a c 0…",null,null,null,null,null,"""1""","""0""","""Napiš správně dě, tě, ně, bě, …","""křížovky, osmisměrky, doplňova…","""Zuzana Pospíšilová ; ilustrova…",null,null,null,null,null,"[""112 stran :""]","[""ilustrace ;""]","[""29 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""pracovní sešity"", ""publikace pro děti"", … ""children's literature""]","[""fd133116"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Trsťan, Drahomír,""","""ill""","[""1958-""]","""mzk2010598344""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2019,112,"""Zuzana Pospíšilová""","""Drahomír Trsťan""","""Pospíšilová & Trsťan"""
"""1""","""Pospíšilová, Zuzana,""","""mzk2006331486""","[""aut""]","""1975-""",null,null,null,null,"""nkc20102158419""",""" nam a22 a 4500""","""110325s2011 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Máš nadání na hádání""",null,"""Zuzana Pospíšilová ; ilustrace…",null,null,null,null,null,"[""118 s. :""]","[""barev. il. ;""]","[""23 cm""]",null,null,null,"[""7"", ""9""]","[""publikace pro děti"", ""children's literature""]","[""fd133156"", null]","[""czenas"", ""eczen

In [39]:
dvojice_sort

['105× Štíplová & Němeček',
 '62× Švandrlík & Winter-Neprakta',
 '30× Pospíšilová & Trsťan',
 '29× Hašek & Lada',
 '27× Rosecká & Růžička']

In [140]:
base_ill = alt.Chart(
    alt_friendly(do_grafu), 
    title=alt.TitleParams(f"{len(dvojice_aut_ill)} nejčastějších dvojic autor/ka-ilustrátor/ka (bez reprintů)")).mark_circle(size=7, filled=True) 

tecky_ill = base_ill.encode(
            x=alt.X("rok:T", title=None, axis=alt.Axis(domainOpacity=0, tickColor='#DCDDD6')), 
            y=alt.Y("dvojice:N", sort=dvojice_sort, title=None, axis=alt.Axis(orient='left', domainOpacity=0, tickColor='white', labelExpr='split(datum.label, "× ")[1]')), 
            yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(range=[3, 15])), 
            color=alt.Color('dvojice:N', scale=alt.Scale(range=['#E09DA3']), 
                            sort=dvojice_sort).legend(None)) \
        .transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")

pocty_ill = base_ill.mark_text().encode(x=alt.X('rok:T', title=None), 
    y=alt.Y('dvojice:N', title=None, sort=dvojice_sort, axis=alt.Axis(orient="right", tickColor='white', labelExpr='split(datum.label, " ")[0]')))

zebricek_ill = alt.layer(tecky_ill, pocty_ill).configure_view(stroke='transparent').properties(
    width=kredity['sirka'] * 1.12, 
    autosize={'type': 'fit', 'contains': 'padding'}
)

zebricek_ill

alt.LayerChart(...)

In [41]:
do_grafu.filter(pl.col('100_a').str.contains('Hašek')).select(pl.col("245_a")).to_series().to_list()

['Dva tucty povídek',
 'Ze staré droguerie',
 'Všivá historie a jiné humoresky',
 'Veselé povídky',
 'Povídky',
 'Když kvetou třešně a jiné humoresky',
 'Procházka přes hranice',
 'Malá zoologická zahrada',
 'Za války i za sovětů v Rusku',
 'Turista Aratáš a jiné humoresky',
 'Pod věchýtkem humoru',
 'Oslí historie, aneb, Vojenské články do čítanek',
 'Má drahá přítelkyně Julča',
 'Reelní podnik',
 'Potměšilé historie',
 'Humoresky',
 'Smějeme se s Jaroslavem Haškem',
 'Nešťastný policejní ředitel',
 'Dobrý voják Švejk před válkou a jiné podivné historky',
 'Když bolševici zrušili Vánoce',
 'Osudy dobrého vojáka Švejka',
 'Švejk před světovou válkou, Velitelem města Bugulmy a další příběhy',
 'Zpověď starého mládence',
 'Hašek v kostce',
 'Aféra s křečkem a jiné povídky',
 'Postrach domu',
 'Dekameron humoru a satiry',
 'Průvodčí cizinců a jiné satiry z cest i z domova',
 'Osudy dobrého vojáka Švejka za světové války',
 'Útrapy vychovatele']

In [142]:
me_to_neurazi(zebricek_ill, soubor="02_psali_ilustrovali", kredity=kredity['default'])

<figure>
    <a href="https://data.irozhlas.cz/knihy-grafy/02_psali_ilustrovali.svg" target="_blank">
    <img src="https://data.irozhlas.cz/knihy-grafy/02_psali_ilustrovali.svg" width="100%" alt="Omlouváme se, ale alternativní text se nepodařilo vygenerovat. Texty v grafu by měly být čitelné ze zdrojového souboru SVG." />
    </a>
    </figure>


In [43]:
df_ill.group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

100_a,100_7,700_a,245_a
str,str,str,u32
"""Štíplová, Ljuba,""","""jk01131441""","""Němeček, Jaroslav,""",108
"""Nedbalová, Marie""","""jo2014816080""","""Popprová, Andrea""",87
"""Nedbalová, Marie""","""jo2014816080""","""Poppr, Roman""",68
"""Bass, Eduard,""","""jk01011066""","""Kratochvíl, Zdeněk,""",67
"""Švandrlík, Miloslav,""","""jk01131832""","""Winter-Neprakta, Jiří,""",64
"""Wilson, Jacqueline,""","""jn20010310318""","""Sharratt, Nick,""",52
"""Delahaye, Gilbert,""","""xx0203193""","""Marlier, Marcel,""",51
"""Verne, Jules,""","""jn19990008769""","""Benett, Léon,""",50
"""Brezina, Thomas,""","""jn20001227589""","""Fearn, Naomi,""",43


In [44]:
ctyrlistek_leader = " <leader>     cam a22      a 4500"
ctyrlistek_leader = ctyrlistek_leader.split(">")[1]
print(ctyrlistek_leader[6])
print(ctyrlistek_leader[7])

a
m


In [45]:
df.filter(pl.col('700_4') == 'ill').filter(~pl.col("100_7").is_in(nechcemejetam)).group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True).rename({'100_a':'autor/ka','700_a':'ilustrátor/ka','245_a':'společných knih'}).drop("100_7")

autor/ka,ilustrátor/ka,společných knih
str,str,u32
"""Štíplová, Ljuba,""","""Němeček, Jaroslav,""",108
"""Nedbalová, Marie""","""Popprová, Andrea""",87
"""Nedbalová, Marie""","""Poppr, Roman""",68
"""Bass, Eduard,""","""Kratochvíl, Zdeněk,""",67
"""Švandrlík, Miloslav,""","""Winter-Neprakta, Jiří,""",64
"""Čapek, Karel,""","""Čapek, Josef,""",33
"""Pospíšilová, Zuzana,""","""Trsťan, Drahomír,""",31
"""Hašek, Jaroslav,""","""Lada, Josef,""",29
"""Rosecká, Zdena""","""Růžička, Jiří""",28


In [46]:
df.filter(pl.col('700_4') == 'trl').group_by(['100_a','100_7','700_a']).agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

100_a,100_7,700_a,245_a
str,str,str,u32
"""Unger, Gert F.,""","""jn20001103529""","""Butala, Tomáš,""",311
"""Kirby, John""","""jx20040611003""","""Pavka, Marek,""",182
"""Courths-Mahler, Hedwig,""","""jn19990001513""","""Schönová, Zuzana""",120
"""Scott, William,""","""ola2003188680""","""Butala, Tomáš,""",101
"""Birkner-Mahler, Frieda,""","""jn20000600873""","""Lacinová, Libuše,""",80
"""Unger, Gert F.,""","""jn20001103529""","""Schönová, Zuzana""",80
"""Unger, Gert F.,""","""jn20001103529""","""Sedláčková, Libuše""",77
"""Pratchett, Terry,""","""jo20000080627""","""Kantůrek, Jan,""",57
"""Brezina, Thomas,""","""jn20001227589""","""Steidlová, Dagmar,""",55


In [47]:
df.filter(pl.col('700_4') == 'ill').group_by('700_a').agg(pl.col("100_a").n_unique()).sort(by="100_a",descending=True)

700_a,100_a
str,u32
"""Born, Adolf,""",149
"""Bouda, Cyril,""",114
"""Burian, Zdeněk,""",112
"""Krejčová, Zdeňka,""",98
"""Svolinský, Karel,""",93
"""Zmatlíková, Helena,""",80
"""Aleš, Mikoláš,""",80
"""Petráček, Jiří,""",78
"""Jiránek, Vladimír,""",75


In [48]:
nejaktivnejsi_ilustratori = df.filter(pl.col("stran") > 30).filter(pl.col('700_4') == 'ill').group_by('700_7').agg(pl.struct(["100_a","245_a"]).n_unique()).sort(by="100_a",descending=True).head(11).select(pl.col("700_7")).to_series().to_list()
nejaktivnejsi_ilustratori = [x for x in nejaktivnejsi_ilustratori if x != None]
nejaktivnejsi_ilustratori

['jk01012660',
 'jk01020396',
 'jk01083186',
 'jk01083128',
 'jk01012795',
 'jk01063265',
 'jk01152754',
 'jk01021645',
 'jk01132366',
 'ola2003162788']

In [49]:
def hezkejmeno(sto):
    if not sto[-1].isalnum():
        sto = sto[:-1]
    if "," in sto:
        sto = sto.split(",")
        sto = sto[1].strip() + " " + sto[0].strip()
    return sto    

In [50]:
do_grafu2 = df.filter(pl.col('700_7').is_in(nejaktivnejsi_ilustratori)).with_columns(pl.col('700_a').map_elements(hezkejmeno)).with_columns(pl.col("rok").map_elements(lambda x: datetime.date(year=int(x), month=1, day=1), return_dtype=pl.Date).cast(pl.Datetime))
nejaktivnejsi_ilustratori2 = do_grafu2.group_by('700_a').agg(pl.struct(["100_a","245_a"]).n_unique()).sort(by="100_a",descending=True).head(10).select(pl.col("700_a")).to_series().to_list()
mrtvi_ilustratori = aut.explode("100_7").filter(pl.col('100_7').is_in(nejaktivnejsi_ilustratori)).explode("046_g").with_columns(pl.col("046_g").map_elements(lambda x: int(x)).alias("umrti")).select(pl.col(["100_7","umrti"])).filter(pl.col('umrti').is_between(1800,2025)).with_columns(pl.col("umrti").map_elements(lambda x: datetime.date(year=int(x), month=1, day=1), return_dtype=pl.Date).cast(pl.Datetime))
do_grafu2 = do_grafu2.join(mrtvi_ilustratori, how='left', left_on='700_7', right_on='100_7').join(
    do_grafu2.group_by('700_7').len(), on='700_7', how='left'
).with_columns(
    pl.col("len").map_elements(lambda x: str(x) + "×")
).with_columns(
    pl.concat_str([pl.col('len'), pl.col('700_a')], separator=' ').alias('700_a')
)
nejaktivnejsi_ilustratori2 = do_grafu2.group_by('700_a').len().sort(by='len',descending=True).select(pl.col('700_a')).to_series().to_list()

In [51]:
mrtvi_ilustratori

100_7,umrti
str,datetime[μs]
"""jk01012660""",2016-01-01 00:00:00
"""jk01012795""",1984-01-01 00:00:00
"""jk01020396""",1981-01-01 00:00:00
"""jk01021645""",1936-01-01 00:00:00
"""jk01132366""",1917-01-01 00:00:00
"""jk01083186""",2011-01-01 00:00:00
"""jk01152754""",2005-01-01 00:00:00


In [52]:
nejaktivnejsi_ilustratori2

['457× Zdeněk Burian',
 '451× Helena Zmatlíková',
 '445× Adolf Born',
 '254× Antonín Šplíchal',
 '252× Cyril Bouda',
 '233× Jiří Winter-Neprakta',
 '204× Věnceslav Černý',
 '198× Zdeňka Krejčová',
 '197× Jaroslav Němeček',
 '169× Karel Ladislav Thuma']

In [53]:
do_grafu2.sample(5)

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,umrti,len
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],datetime[μs],i64,datetime[μs],str
"""1""","""Brukner, Josef,""","""jk01013304""","[""aut""]","""1932-2015""",null,null,null,null,"""cpk19970216476""",""" nam a22 a 4500""","""970922s1997 xr ag b p 0…",null,null,null,null,null,"""1""","""0""","""Čítanka pro 4. ročník základní…","""knížka ke čtení, zpívání, hran…","""Josef Brukner, Miroslava Čížko…",null,null,null,null,null,"[""224 s. :""]","[""barev. il., noty ;""]","[""21 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""čítanky"", ""učebnice základních škol"", … ""textbooks (elementary)""]","[""fd133984"", ""fd133773"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1"", ""1"", ""1""]","""198× Zdeňka Krejčová""","""ill""","[null, null, ""1944-""]","""jk01063265""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1997-01-01 00:00:00,224,null,"""198×"""
"""1""","""Doucha, František,""","""jk01022897""","[""aut""]","""1810-1884""",null,null,null,null,"""cpk20021190978""",""" nam a22 a 4500""","""021031s1885 xr ac c 0…",null,null,null,null,null,"""1""","""0""","""Několik perliček""",null,"""od Františka Douchy ; illustro…",null,null,null,null,null,"[""46 s. :""]","[""il., portrét ;""]","[""17 cm""]",null,null,null,"[""7"", ""7"", … ""7""]","[""didaktické povídky"", ""publikace pro mládež"", … ""česká poezie""]","[""fd165100"", ""fd133157"", … ""fd133958""]","[""czenas"", ""czenas"", … ""czenas""]",null,null,null,"[""1""]","""169× Karel Ladislav Thuma""","""ill""","[""1853-1917""]","""jk01132366""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1885-01-01 00:00:00,46,1917-01-01 00:00:00,"""169×"""
"""1""","""Seidel, Ina,""","""xx0022624""","[""aut""]","""1885-1974""",null,null,null,null,"""bk194103284""",""" nam a22 1 4500""","""990322s1941 xr …",null,null,null,null,null,"""1""","""0""","""Mezi dvěma válkami""","""[Das Wunschkind] : 1792-1813.""","""Ina Seidel ; [z němčiny přelož…","[""I. díl /""]",null,null,null,null,"[""423, [I] s. ;""]",null,"[""8°""]",null,null,null,null,null,null,null,null,null,null,"[""1"", ""1""]","""252× Cyril Bouda""","""bkd""","[""1911-2000"", ""1901-1984""]","""jk01012795""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1941-01-01 00:00:00,423,1984-01-01 00:00:00,"""252×"""
"""1""","""Rodari, Gianni,""","""jn19990007094""","[""aut""]","""1920-1980""",null,null,null,null,"""nkc20162812347""",""" nam a22 i 4500""","""160712s2016 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Pohádky na hraní""",null,"""Gianni Rodari ; ilustroval Ado…",null,null,null,null,null,"[""125 stran :""]","[""barevné ilustrace ;""]","[""25 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""italské příběhy"", ""publikace pro děti"", … ""children's literature""]","[""fd183393"", ""fd133156"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1"", ""1""]","""445× Adolf Born""","""ill""","[""1930-2016"", ""1978-""]","""jk01012660""",null,null,nul

In [54]:
nejaktivnejsi_ilustratori2

['457× Zdeněk Burian',
 '451× Helena Zmatlíková',
 '445× Adolf Born',
 '254× Antonín Šplíchal',
 '252× Cyril Bouda',
 '233× Jiří Winter-Neprakta',
 '204× Věnceslav Černý',
 '198× Zdeňka Krejčová',
 '197× Jaroslav Němeček',
 '169× Karel Ladislav Thuma']

In [55]:
do_grafu2.sample(3)

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran,umrti,len
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],datetime[μs],i64,datetime[μs],str
"""1""","""Uzel, Radim,""","""jk01140932""","[""aut""]","""1940-2022""",null,null,null,null,"""nkc20142633424""",""" nam a22 a 4500""","""141016s2014 xr a e f 0…",null,null,null,null,null,"""1""","""0""","""Sexuální všehochuť""","""(podle abecedy) /""","""Radim Uzel ; [ilustrace Jiří W…",null,null,null,null,null,"[""206 s. :""]","[""il. ;""]","[""22 cm""]",null,null,null,"[""7"", ""9""]","[""populárně-naučné publikace"", ""popular works""]","[""fd131864"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1""]","""233× Jiří Winter-Neprakta""","""ill""","[""1924-2011""]","""jk01083186""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2014-01-01 00:00:00,206,2011-01-01 00:00:00,"""233×"""
"""1""","""Baar, Jindřich Šimon,""","""jk01010468""","[""aut""]","""1869-1925""",null,null,null,null,"""bk197504488""",""" nam a22 1 4500""","""971013s1975 xr g 0…",null,null,null,null,null,"""1""","""0""","""Osmačtyřicátníci""",null,"""Jindřich Šimon Baar ; kresby: …",null,null,null,null,null,"[""437, [2] s. ;""]",null,"[""8°""]",null,null,null,"[""7"", ""9""]","[""české romány"", ""Czech fiction""]","[""fd133974"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1"", ""1""]","""252× Cyril Bouda""","""ill""","[""1930-2024"", ""1901-1984""]","""jk01012795""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1975-01-01 00:00:00,437,1984-01-01 00:00:00,"""252×"""
"""1""","""Horníček, Miroslav,""","""jk01042235""","[""aut""]","""1918-2003""",null,null,null,null,"""ck8404600""",""" nam a22 4500""","""841228s1984 xr a u0…",null,null,null,null,null,"""1""","""0""","""Dobře utajené housle""","""Jablko je vinno /""","""Miroslav Horníček ; [ilustrace…",null,null,null,null,null,"[""102, 170 s. :""]","[""il. ;""]","[""21 cm""]",null,null,null,null,null,null,null,null,null,null,"[""1""]","""445× Adolf Born""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1984-01-01 00:00:00,170,2016-01-01 00:00:00,"""445×"""


In [56]:
base = alt.Chart(
    do_grafu2.filter(pl.col('700_a').is_in(nejaktivnejsi_ilustratori2)).to_pandas(), title=alt.TitleParams(
        f"{len(nejaktivnejsi_ilustratori2)} nejvydávanějších ilustrátorů a ilustrátorek (bez reprintů)",
    subtitle="Co tečka, to kniha. Černá čárka označuje rok úmrtí."))

kulicky = base.mark_circle(size=7, filled=True).encode(
            x=alt.X("rok:T", title=None, axis=alt.Axis(domainOpacity=0, tickColor='#DCDDD6')), 
            y=alt.Y("700_a:N", sort=nejaktivnejsi_ilustratori2, title=None, axis=alt.Axis(orient='left', domainOpacity=0, tickColor='white', labelExpr='split(datum.label, "× ")[1]' )), #, 
            yOffset=alt.YOffset("jitter:Q", scale=alt.Scale(range=[3, 15])), 
            color=alt.Color('700_a:N', scale=alt.Scale(range=['#E09DA3']), 
                            sort=nejaktivnejsi_ilustratori2).legend(None)) \
        .transform_calculate(jitter="sqrt(-2*log(random()))*cos(2*PI*random())")

kdy_umreli = base.mark_tick(
    color='#292829',  # optional: you can specify color
    thickness=1.5,
    height=9
).encode(
    x=alt.X('umrti:T', title=None),
    y=alt.Y("700_a:N", sort=nejaktivnejsi_ilustratori2, title=None, axis=alt.Axis(orient='left', tickColor='white', labels=False)))

pocty = base.mark_text().encode(x=alt.X('rok:T', title=None), 
    y=alt.Y('700_a:N', title=None, sort=nejaktivnejsi_ilustratori2, axis=alt.Axis(orient="right", tickColor='white', labelExpr='split(datum.label, " ")[0]'))) #

zebricek2 = alt.layer(kulicky, kdy_umreli, pocty).configure_view(stroke='transparent').properties(
    width=kredity['sirka'] * 1.12,
    autosize={'type': 'fit', 'contains': 'padding'}
).resolve_scale(color='independent',x="shared")

zebricek2

alt.LayerChart(...)

In [57]:
me_to_neurazi(zebricek2, kredity=kredity['default'], soubor='02_ilustratorstvo')

<figure>
    <a href="https://data.irozhlas.cz/knihy-grafy/02_ilustratorstvo.svg" target="_blank">
    <img src="https://data.irozhlas.cz/knihy-grafy/02_ilustratorstvo.svg" width="100%" alt="Omlouváme se, ale alternativní text se nepodařilo vygenerovat. Texty v grafu by měly být čitelné ze zdrojového souboru SVG." />
    </a>
    </figure>


In [58]:
df.filter(pl.col('700_a') == 'Born, Adolf,')

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,022_a,022_y,022_z,022_ind1,022_l,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,300_a,300_b,300_c,300_e,300_f,300_3,655_ind2,655_a,655_7,655_2,655_ind1,655_y,655_z,700_ind1,700_a,700_4,700_d,700_7,700_t,700_q,700_l,700_ind2,700_c,700_b,700_i,700_m,700_k,700_n,700_r,700_p,700_o,700_s,700_j,700_x,700_e,700_f,700_5,700_9,700_g,rok,stran
str,str,str,list[str],str,str,list[str],str,str,str,str,str,str,list[str],list[str],str,str,str,str,str,str,str,list[str],str,str,str,str,list[str],list[str],list[str],list[str],str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],i64,i64
"""1""","""Pecháček, Ladislav,""","""jk01092311""","[""aut""]","""1940-""",null,null,null,null,"""cpk20041296787""",""" nam a22 a 4500""","""040329s2004 xr af e 0…",null,null,null,null,null,"""1""","""0""","""Jak básníkům chutná život""",null,"""Ladislav Pecháček ; [ilustrace…",null,null,null,null,null,"[""133 s., [8.] s. obr. příl. :""]","[""il. ;""]","[""19 cm""]",null,null,null,"[""7"", ""9""]","[""české povídky"", ""Czech short stories""]","[""fd133971"", null]","[""czenas"", ""eczenas""]",null,null,null,"[""1""]","""Born, Adolf,""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2004,133
"""1""","""Branald, Adolf,""","""jk01012955""","[""aut""]","""1910-2008""",null,null,null,null,"""ck9203967""",""" nam a22 4500""","""920430s1992 xr a u0…",null,null,null,null,null,"""1""","""0""","""Báječní muži na okřídlených oř…",null,"""Adolf Branald ; [ilustrace Ado…",null,null,null,null,null,"[""319 s. :""]","[""il. ;""]","[""21 cm""]",null,null,null,"[""7""]","[""eseje""]","[""fd132213""]","[""czenas""]",null,null,null,"[""1""]","""Born, Adolf,""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1992,319
"""1""","""Drijverová, Martina,""","""jk01023067""","[""aut""]","""1951-2022""",null,null,null,null,"""ck8902252""",""" nam a22 4500""","""890311s1988 xr a u0…",null,null,null,null,null,"""1""","""0""","""Sísa Kyselá""","""pro začínající čtenáře /""","""Martina Drijverová ; ilustrace…",null,null,null,null,null,"[""59 s. :""]","[""barev. il. ;""]","[""21 cm""]",null,null,null,null,null,null,null,null,null,null,"[""1""]","""Born, Adolf,""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1988,59
"""1""","""Hercíková, Iva,""","""jk01040797""","[""aut""]","""1935-2007""",null,null,null,null,"""cpk19990659553""",""" nam a22 a 4500""","""990518s1999 xr a d 0…",null,null,null,null,null,"""1""","""0""","""Druhá láska""",null,"""Iva Hercíková ; [ilustroval Ad…",null,null,null,null,null,"[""141 s. :""]","[""il. ;""]","[""21 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""milostné romány"", ""publikace pro mládež"", … ""Czech fiction""]","[""fd132840"", ""fd133157"", … null]","[""czenas"", ""czenas"", … ""eczenas""]",null,null,null,"[""1""]","""Born, Adolf,""","""ill""","[""1930-2016""]","""jk01012660""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1999,141
"""1""","""Drijverová, Martina,""","""jk01023067""","[""aut""]","""1951-2022""",null,null,null,null,"""nkc20233547511""",""" nam a22 i 4500""","""230911s2023 xr a b 0…",null,null,null,null,null,"""1""","""0""","""Sísa Kyselá""",null,"""Martina Drijverová ; ilustrova…",null,null,null,null,null,"[""62 stran :""]","[""barevné ilustrace ;""]","[""21 cm""]",null,null,null,"[""7"", ""7"", … ""9""]","[""české příběhy"", 

In [59]:
df.filter(pl.col('700_4') == 'ill').group_by('700_a').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

700_a,245_a
str,u32
"""Born, Adolf,""",281
"""Burian, Zdeněk,""",230
"""Němeček, Jaroslav,""",179
"""Bouda, Cyril,""",166
"""Zmatlíková, Helena,""",164
"""Winter-Neprakta, Jiří,""",160
"""Krejčová, Zdeňka,""",159
"""Šplíchal, Antonín,""",143
"""Thuma, Karel Ladislav,""",134


In [60]:
df_preklady = df.join(pl.read_parquet(os.path.join("data/cnb_sloupce","041.parquet")), left_on="001", right_on="001", how="left")

In [61]:
df_preklady.filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

700_a,245_a
str,u32
"""Butala, Tomáš,""",502
"""Pavka, Marek,""",429
"""Vyskočil, Josef""",344
"""Sladká, Stanislava""",246
"""Schönová, Zuzana""",234
"""Volhejnová, Veronika,""",230
"""Podaný, Richard,""",224
"""Kuťák, Jaroslav,""",214
"""Chromiaková, Eva""",196


In [62]:
df_preklady.filter(pl.col('700_a') == "Butala, Tomáš,").group_by("100_a").len()

100_a,len
str,u32
"""Taylor, Kathryn""",3
"""Aurel, Catherine,""",1
"""Goga-Klinkenberg, Susanne,""",4
"""Unger, Gert F.,""",391
"""Roberts, Dan,""",1
"""Ryan, William""",4
"""Murphy, Bill,""",10
"""Denver, Jeff,""",1
"""Schröder, Rainer M.,""",1


In [63]:
df_preklady.filter(pl.col('700_a') == "Butala, Tomáš,").group_by("041_h").len()

041_h,len
list[str],u32
"[""ice""]",5
"[""fre""]",1
null,2
"[""ger""]",633
"[""eng""]",3


In [64]:
df_preklady.filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

700_a,245_a
str,u32
"""Butala, Tomáš,""",502
"""Pavka, Marek,""",429
"""Vyskočil, Josef""",344
"""Sladká, Stanislava""",246
"""Schönová, Zuzana""",234
"""Volhejnová, Veronika,""",230
"""Podaný, Richard,""",224
"""Kuťák, Jaroslav,""",214
"""Chromiaková, Eva""",196


In [65]:
df_preklady.filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("stran").sum()).sort(by="stran",descending=True)

700_a,stran
str,i64
"""Volhejnová, Veronika,""",104086
"""Pacnerová, Jana,""",90284
"""Kantůrek, Jan,""",80551
"""Podaný, Richard,""",76323
"""Dušek, Zdík,""",68870
"""Jašová, Jana,""",66951
"""Klůfová, Petra,""",66769
"""Chodilová, Dana,""",56060
"""Medek, Pavel,""",54195


In [66]:
df_preklady.filter(pl.col('700_a') == "Volhejnová, Veronika,").group_by("100_a").len().sort(by="len",descending=True)

100_a,len
str,u32
"""Kinney, Jeff,""",52
"""Christie, Agatha,""",29
"""Herbert, Frank,""",24
"""Lewis, C. S.""",21
"""Walliams, David,""",21
"""Green, John,""",17
"""Colfer, Chris,""",16
"""Le Carré, John,""",8
"""Morse, Brian,""",7


In [67]:
df_preklady.explode("041_h").filter(pl.col('700_4') == 'trl').group_by('700_a').agg(pl.col("041_h").n_unique()).sort(by="041_h",descending=True)

700_a,041_h
str,u32
"""Babler, Otto František,""",19
"""Vetti, O. S.,""",16
"""Hiršal, Josef,""",16
"""Vrchlický, Jaroslav,""",12
"""Bednář, Kamil,""",12
"""Žáček, Jiří,""",11
"""Sýs, Karel,""",11
"""Vladislav, Jan,""",11
"""Fischer, Otokar,""",11


In [68]:
df_preklady.filter(pl.col('700_a') == "Babler, Otto František,").group_by('041_h').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

041_h,245_a
list[str],u32
"[""eng""]",10
"[""ger""]",9
"[""fre""]",8
"[""scr""]",5
null,5
"[""hrv""]",3
"[""ita""]",3
"[""bul""]",2
"[""lat""]",2


In [69]:
df_preklady.filter(pl.col('700_a') == "Hiršal, Josef,").group_by('041_h').agg(pl.col("245_a").n_unique()).sort(by="245_a",descending=True)

041_h,245_a
list[str],u32
"[""ger""]",32
null,12
"[""por""]",4
"[""spa""]",3
"[""mul""]",2
"[""scr""]",2
"[""rum""]",2
"[""swe""]",2
"[""fre""]",2
